# K-Means Clustering — from scratch

K-Means partitions $n$ points into $k$ clusters by minimising the **within-cluster sum of squares (inertia)**:

$$J = \sum_{j=1}^{k}\sum_{\mathbf{x}\in C_j}\lVert\mathbf{x} - \boldsymbol{\mu}_j\rVert^2$$

**Lloyd's algorithm**
1. Initialise $k$ centroids.
2. **Assignment step**: assign each point to its nearest centroid.
3. **Update step**: move each centroid to the mean of its assigned points.
4. Repeat 2–3 until assignments stop changing (or centroids move less than a tolerance).

Each step can only decrease $J$, so the algorithm converges — but to a **local** minimum that depends on initialisation.

**K-Means++ initialisation**: choose the first centre uniformly at random; choose each next centre with probability
proportional to $D(\mathbf{x})^2$, the squared distance to the nearest centre already chosen. This spreads the initial centres out.

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

# NumPy 2.0 with Apple's Accelerate BLAS raises spurious "encountered in matmul" warnings on macOS
warnings.filterwarnings("ignore", message=".*encountered in matmul")

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 5)

In [ ]:
def make_blobs(n_samples=300, centers=3, cluster_std=1.0, n_features=2,
               center_box=(-10, 10), random_state=None):
    """Isotropic Gaussian blobs (re-implementation of sklearn's make_blobs)."""
    rng = np.random.RandomState(random_state)
    if isinstance(centers, int):
        centers = rng.uniform(*center_box, size=(centers, n_features))
    centers = np.asarray(centers, dtype=float)
    k = len(centers)
    counts = [n_samples // k + (1 if i < n_samples % k else 0) for i in range(k)]
    stds = np.broadcast_to(np.asarray(cluster_std, dtype=float), (k,))
    X = np.vstack([rng.normal(c, s, size=(m, centers.shape[1]))
                   for c, s, m in zip(centers, stds, counts)])
    y = np.concatenate([np.full(m, i) for i, m in enumerate(counts)])
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

def make_moons(n_samples=200, noise=0.1, random_state=None):
    """Two interleaving half circles."""
    rng = np.random.RandomState(random_state)
    n_out = n_samples // 2
    n_in = n_samples - n_out
    t_out = np.linspace(0, np.pi, n_out)
    t_in = np.linspace(0, np.pi, n_in)
    outer = np.c_[np.cos(t_out), np.sin(t_out)]
    inner = np.c_[1 - np.cos(t_in), 0.5 - np.sin(t_in)]
    X = np.vstack([outer, inner]) + rng.normal(0, noise, size=(n_samples, 2))
    y = np.r_[np.zeros(n_out, dtype=int), np.ones(n_in, dtype=int)]
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

In [ ]:
def pairwise_distances(A, B):
    sq = (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None, :] - 2 * A @ B.T
    return np.sqrt(np.maximum(sq, 0))


def silhouette_score(X, labels):
    """Mean of s(i) = (b - a) / max(a, b); a = mean intra-cluster distance,
    b = mean distance to the nearest other cluster. Noise points (label -1) are ignored."""
    mask = labels != -1
    X, labels = X[mask], labels[mask]
    clusters = np.unique(labels)
    if len(clusters) < 2:
        return np.nan
    D = pairwise_distances(X, X)
    s = np.zeros(len(X))
    for i in range(len(X)):
        same = labels == labels[i]
        if same.sum() == 1:
            continue                                   # singleton cluster: s(i) = 0
        a = D[i, same].sum() / (same.sum() - 1)
        b = min(D[i, labels == c].mean() for c in clusters if c != labels[i])
        s[i] = (b - a) / max(a, b)
    return s.mean()

## Implementation

In [ ]:
class KMeans:
    def __init__(self, n_clusters=3, init="k-means++", n_init=10, max_iter=300, tol=1e-6, random_state=None):
        self.k = n_clusters
        self.init = init
        self.n_init = n_init
        self.max_iter = max_iter
        self.tol = tol
        self.rng = np.random.RandomState(random_state)

    def _init_centroids(self, X):
        n = len(X)
        if self.init == "random":
            return X[self.rng.choice(n, self.k, replace=False)].copy()
        centers = [X[self.rng.randint(n)]]
        for _ in range(1, self.k):
            d2 = np.min(((X[:, None, :] - np.array(centers)[None]) ** 2).sum(-1), axis=1)
            centers.append(X[self.rng.choice(n, p=d2 / d2.sum())])
        return np.array(centers)

    @staticmethod
    def _assign(X, centers):
        d2 = ((X[:, None, :] - centers[None]) ** 2).sum(-1)
        return np.argmin(d2, axis=1), d2.min(axis=1).sum()

    def _single_run(self, X):
        centers = self._init_centroids(X)
        history = [centers.copy()]
        for it in range(self.max_iter):
            labels, _ = self._assign(X, centers)
            new_centers = np.array([X[labels == j].mean(0) if np.any(labels == j)
                                    else X[self.rng.randint(len(X))]          # re-seed an empty cluster
                                    for j in range(self.k)])
            shift = np.sum((new_centers - centers) ** 2)
            centers = new_centers
            history.append(centers.copy())
            if shift < self.tol:
                break
        labels, inertia = self._assign(X, centers)
        return centers, labels, inertia, it + 1, history

    def fit(self, X):
        best = None
        for _ in range(self.n_init):                      # keep the run with the lowest inertia
            run = self._single_run(X)
            if best is None or run[2] < best[2]:
                best = run
        self.cluster_centers_, self.labels_, self.inertia_, self.n_iter_, self.history_ = best
        return self

    def predict(self, X):
        return self._assign(X, self.cluster_centers_)[0]

In [ ]:
X, y_true = make_blobs(n_samples=500, centers=4, cluster_std=1.2, random_state=10)
km = KMeans(n_clusters=4, random_state=0).fit(X)
print(f"inertia = {km.inertia_:.2f}, iterations = {km.n_iter_}, silhouette = {silhouette_score(X, km.labels_):.3f}")

plt.scatter(X[:, 0], X[:, 1], c=km.labels_, cmap="viridis", s=12)
plt.scatter(*km.cluster_centers_.T, c="red", marker="X", s=250, edgecolors="k", label="centroids")
plt.legend(); plt.title("K-Means (k=4)"); plt.show()

## Watching Lloyd's algorithm converge (random initialisation)

In [ ]:
km_vis = KMeans(n_clusters=4, init="random", n_init=1, random_state=3).fit(X)
steps = [0, 1, 2, len(km_vis.history_) - 1]
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, s in zip(axes, steps):
    centers = km_vis.history_[s]
    labels, J = KMeans._assign(X, centers)
    ax.scatter(X[:, 0], X[:, 1], c=labels, cmap="viridis", s=8)
    path = np.array(km_vis.history_[: s + 1])
    for j in range(4):
        ax.plot(path[:, j, 0], path[:, j, 1], "r.-")
    ax.scatter(*centers.T, c="red", marker="X", s=200, edgecolors="k")
    ax.set_title(f"iteration {s}, inertia = {J:.0f}")
plt.tight_layout(); plt.show()

## Random vs K-Means++ initialisation

In [ ]:
Xb, _ = make_blobs(n_samples=600, centers=8, cluster_std=0.6, random_state=5)
results = {}
for init in ["random", "k-means++"]:
    results[init] = [KMeans(8, init=init, n_init=1, random_state=s).fit(Xb).inertia_ for s in range(50)]
    print(f"{init:10s}: mean inertia = {np.mean(results[init]):8.1f}, worst = {np.max(results[init]):8.1f}, "
          f"best = {np.min(results[init]):.1f}")
plt.boxplot(list(results.values()))
plt.xticks([1, 2], list(results.keys()))
plt.ylabel("final inertia (50 single runs)"); plt.title("Initialisation matters"); plt.show()

## Choosing $k$: elbow method and silhouette score
Inertia always decreases with $k$; look for the "elbow". The silhouette score peaks at a good $k$.

In [ ]:
ks = range(2, 11)
inertias, sils = [], []
for k in ks:
    m = KMeans(k, random_state=0).fit(X)
    inertias.append(m.inertia_)
    sils.append(silhouette_score(X, m.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(ks, inertias, "o-"); axes[0].set_xlabel("k"); axes[0].set_ylabel("inertia"); axes[0].set_title("Elbow method")
axes[1].plot(ks, sils, "s-", c="tab:green"); axes[1].set_xlabel("k"); axes[1].set_ylabel("silhouette")
axes[1].set_title(f"Silhouette (best k = {list(ks)[int(np.argmax(sils))]})")
plt.tight_layout(); plt.show()

## Limitations
K-Means assumes convex, isotropic clusters of similar size. It struggles with non-convex shapes and unequal variances.

In [ ]:
Xm, _ = make_moons(400, noise=0.05, random_state=0)
Xv, _ = make_blobs(450, centers=[[-6, 0], [0, 0], [4, 0]], cluster_std=[0.4, 3.0, 0.4], random_state=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, (Xd, k, title) in zip(axes, [(Xm, 2, "non-convex (moons)"), (Xv, 3, "unequal variance")]):
    m = KMeans(k, random_state=0).fit(Xd)
    ax.scatter(Xd[:, 0], Xd[:, 1], c=m.labels_, cmap="viridis", s=10)
    ax.scatter(*m.cluster_centers_.T, c="red", marker="X", s=200, edgecolors="k")
    ax.set_title(f"K-Means on {title}")
plt.tight_layout(); plt.show()

## Application: colour quantisation
Cluster the pixel colours of an image into $k$ colours and replace each pixel with its centroid.
(The image is generated synthetically so the notebook has no external files.)

In [ ]:
h, w = 120, 180
yy, xx = np.mgrid[0:h, 0:w] / np.array([h, w])[:, None, None]
img = np.dstack([0.5 + 0.5 * np.sin(6 * xx), 0.5 + 0.5 * np.cos(5 * yy + 2 * xx), xx * yy])
img += np.random.RandomState(0).normal(0, 0.03, img.shape)
img = np.clip(img, 0, 1)
pixels = img.reshape(-1, 3)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
axes[0].imshow(img); axes[0].set_title("original"); axes[0].axis("off")
sample = pixels[np.random.RandomState(1).choice(len(pixels), 3000, replace=False)]   # fit on a sample for speed
for ax, k in zip(axes[1:], [2, 4, 8]):
    m = KMeans(k, n_init=3, random_state=0).fit(sample)
    ax.imshow(m.cluster_centers_[m.predict(pixels)].reshape(img.shape)); ax.set_title(f"k = {k} colours"); ax.axis("off")
plt.tight_layout(); plt.show()

## Summary
* Simple and fast: $O(n \cdot k \cdot d)$ per iteration.
* Needs $k$ in advance (elbow / silhouette help), is sensitive to initialisation (use k-means++ and several restarts) and to feature scaling.
* Best for compact, roughly spherical clusters of similar size.